# PPO from scratch, step by step

Based on [ericyangyu/PPO-for-Beginners](https://github.com/ericyangyu/PPO-for-Beginners) (Pendulum-v1).
Everything is flat code (no classes, no functions) and every cell prints what it does, so you can watch PPO work.

**The big picture of one PPO iteration:**

1. **Rollout**: play the game with the current actor and record what happened
2. **Rewards-to-go**: for each step, the discounted sum of future rewards
3. **Advantage**: was the action better or worse than the critic expected?
4. **Update**: several gradient steps with the *clipped* objective
5. Repeat

Run the cells in order. Sections 3-9 walk through ONE iteration in detail; section 10 is the real training loop.

## 0. Install and imports

In [ ]:

!pip install gymnasium

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement gymnasium[classic-control] (from versions: none)
ERROR: No matching distribution found for gymnasium[classic-control]


In [3]:
import time

import gymnasium as gym
import numpy as np
import torch
from torch import nn
from torch.distributions import MultivariateNormal
from torch.optim import Adam
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("gymnasium:", gym.__version__)
print("numpy:", np.__version__)

ModuleNotFoundError: No module named 'gymnasium'

## 1. Hyperparameters

| name | meaning |
|---|---|
| `timesteps_per_batch` | how many steps of experience we collect before each update |
| `max_timesteps_per_episode` | max length of one episode |
| `gamma` | discount factor: how much we care about future rewards (0 to 1) |
| `n_updates_per_iteration` | gradient steps on each batch (PPO can safely reuse a batch a few times) |
| `lr` | learning rate |
| `clip` | PPO clip range (epsilon). The policy may only change by about +/- 20% per iteration |

In [ ]:
timesteps_per_batch = 2048
max_timesteps_per_episode = 200
gamma = 0.99
n_updates_per_iteration = 10
lr = 3e-4
clip = 0.2
save_freq = 10
total_timesteps = 500_000   # original repo used 200_000_000 (just stop whenever it converges)

seed = None   # set an int (e.g. 42) for reproducible runs
if seed is not None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    print("seed set to", seed)

print("hyperparameters loaded")

## 2. The environment

`Pendulum-v1`: swing a pendulum up and keep it balanced.
- **Observation** (3 numbers): `cos(angle)`, `sin(angle)`, `angular velocity`
- **Action** (1 number): torque applied, between -2 and 2
- **Reward**: always <= 0, closer to 0 is better (a perfect run is about -150 to -200 per episode with this setup)
- Pendulum never "terminates"; it is cut off after 200 steps (that is `truncated`)

In [ ]:
env = gym.make('Pendulum-v1', render_mode='rgb_array')

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]

print("Observation space:", env.observation_space)
print("Action space     :", env.action_space)
print("obs_dim =", obs_dim, "| act_dim =", act_dim)

# Take one look at the environment
obs, info = env.reset()
print("\nFirst observation:", obs)
print("  cos(angle) = %.3f, sin(angle) = %.3f, angular_velocity = %.3f" % tuple(obs))

# Take one random step
random_action = env.action_space.sample()
obs2, rew, terminated, truncated, info = env.step(random_action)
print("\nRandom action:", random_action)
print("Next observation:", obs2)
print("Reward:", rew, "| terminated:", terminated, "| truncated:", truncated)

In [ ]:
# What does the environment look like?
frame = env.render()
plt.imshow(frame)
plt.axis('off')
plt.title("Pendulum-v1")
plt.show()

## 3. Actor and critic networks

Two separate small networks, both `input -> 64 -> 64 -> output` with ReLU in between:

- **Actor** (the policy): observation -> *mean* of the action
- **Critic**: observation -> one number `V(s)`, the estimated total future reward from that state

Neither network knows anything yet; their weights are random.

In [ ]:
actor = nn.Sequential(
    nn.Linear(obs_dim, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, act_dim),
)
critic = nn.Sequential(
    nn.Linear(obs_dim, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
)

actor_optim = Adam(actor.parameters(), lr=lr)
critic_optim = Adam(critic.parameters(), lr=lr)

print("ACTOR:\n", actor)
print("\nCRITIC:\n", critic)
print("\nActor parameters :", sum(p.numel() for p in actor.parameters()))
print("Critic parameters:", sum(p.numel() for p in critic.parameters()))

## 4. How the actor picks an action

The actor only outputs a **mean**. To explore, we build a Gaussian (normal) distribution centred on that
mean and **sample** from it. The spread is a fixed covariance: variance 0.5 (std about 0.707).

We also record the **log probability** of the sampled action. PPO needs it later to measure how much
the policy has changed.

In [ ]:
cov_var = torch.full(size=(act_dim,), fill_value=0.5)
cov_mat = torch.diag(cov_var)
print("cov_var:", cov_var)
print("cov_mat:", cov_mat)

# Demo on a single observation
obs, _ = env.reset()
obs_tensor = torch.tensor(obs, dtype=torch.float)

mean = actor(obs_tensor)
dist = MultivariateNormal(mean, cov_mat)

print("\nObservation:", obs)
print("Actor's mean action:", mean.detach().numpy())

print("\nSampling 5 actions from the same distribution (notice they differ = exploration):")
for i in range(5):
    a = dist.sample()
    lp = dist.log_prob(a)
    print(f"  action = {a.item(): .4f}   log_prob = {lp.item(): .4f}")

## 5. Step 1 of an iteration: ROLLOUT

First, watch **one episode** step by step (only the first 8 steps are printed).

In [ ]:
obs, _ = env.reset()
ep_rews_demo = []

for ep_t in range(max_timesteps_per_episode):
    mean = actor(torch.tensor(obs, dtype=torch.float))
    dist = MultivariateNormal(mean, cov_mat)
    action = dist.sample()
    log_prob = dist.log_prob(action)

    new_obs, rew, terminated, truncated, _ = env.step(action.detach().numpy())
    done = terminated | truncated
    ep_rews_demo.append(rew)

    if ep_t < 8:
        print(f"t={ep_t:3d} | obs={np.round(obs, 3)} | mean={mean.item(): .3f} | action={action.item(): .3f} "
              f"| log_prob={log_prob.item(): .3f} | reward={rew: .3f}")
    obs = new_obs
    if done:
        break

print("...")
print(f"Episode finished after {ep_t + 1} steps, total reward = {sum(ep_rews_demo):.2f}")
print(f"truncated={truncated}, terminated={terminated}")

Now collect a **whole batch**: keep playing episodes until we have at least `timesteps_per_batch` steps.
Everything is stored in lists, then converted to tensors.

In [ ]:
batch_obs = []         # observations
batch_acts = []        # actions we took
batch_log_probs = []   # log prob of each action (under the policy that took it)
batch_rews = []        # rewards, one list per episode
batch_lens = []        # length of each episode

t = 0
while t < timesteps_per_batch:
    ep_rews = []
    obs, _ = env.reset()

    for ep_t in range(max_timesteps_per_episode):
        t += 1
        batch_obs.append(obs)

        mean = actor(torch.tensor(obs, dtype=torch.float))
        dist = MultivariateNormal(mean, cov_mat)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        action = action.detach().numpy()
        log_prob = log_prob.detach()

        obs, rew, terminated, truncated, _ = env.step(action)
        done = terminated | truncated

        ep_rews.append(rew)
        batch_acts.append(action)
        batch_log_probs.append(log_prob)

        if done:
            break

    batch_lens.append(ep_t + 1)
    batch_rews.append(ep_rews)

print("Episodes collected  :", len(batch_lens))
print("Episode lengths     :", batch_lens)
print("Total timesteps     :", sum(batch_lens))
print("Episode returns     :", [round(sum(r), 1) for r in batch_rews])

# lists -> tensors
batch_obs = torch.tensor(np.array(batch_obs), dtype=torch.float)
batch_acts = torch.tensor(np.array(batch_acts), dtype=torch.float)
batch_log_probs = torch.tensor(np.array(batch_log_probs), dtype=torch.float)

print("\nbatch_obs.shape       :", tuple(batch_obs.shape), " (timesteps, obs_dim)")
print("batch_acts.shape      :", tuple(batch_acts.shape), " (timesteps, act_dim)")
print("batch_log_probs.shape :", tuple(batch_log_probs.shape), " (timesteps,)")

## 6. Step 2: REWARDS-TO-GO

For each timestep, the **discounted sum of all future rewards** in its episode:

$$R_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots$$

We compute it **backwards**, so each step is simply `rtg = reward + gamma * rtg_of_next_step`.

Tiny example first (rewards `[1, 1, 1]`, gamma = 0.5) so you can verify by hand:

In [ ]:
# Hand-checkable example
example_rews = [1, 1, 1]
example_gamma = 0.5
example_rtgs = []
discounted = 0
for r in reversed(example_rews):
    discounted = r + discounted * example_gamma
    example_rtgs.insert(0, discounted)
    print(f"reward={r}  ->  rtg={discounted}")
print("rewards-to-go:", example_rtgs)
print("check: step 0 = 1 + 0.5*1 + 0.25*1 =", 1 + 0.5 + 0.25)

In [ ]:
# The real thing, on our batch
batch_rtgs = []
for ep_rews in reversed(batch_rews):
    discounted_reward = 0
    for rew in reversed(ep_rews):
        discounted_reward = rew + discounted_reward * gamma
        batch_rtgs.insert(0, discounted_reward)
batch_rtgs = torch.tensor(batch_rtgs, dtype=torch.float)

print("batch_rtgs.shape:", tuple(batch_rtgs.shape), "(same number of entries as timesteps)")
print("first 5 rewards-to-go:", batch_rtgs[:5].numpy().round(2))
print("min / mean / max     : %.2f / %.2f / %.2f" % (batch_rtgs.min(), batch_rtgs.mean(), batch_rtgs.max()))

## 7. Step 3: ADVANTAGE

$$A_t = R_t - V(s_t)$$

- `R_t`: what **actually** happened (reward-to-go)
- `V(s_t)`: what the **critic expected**
- **A > 0**: better than expected -> make this action *more* likely
- **A < 0**: worse than expected -> make this action *less* likely

We then **normalise** (mean 0, std 1). That is not in the theory, but it makes training much more stable.

In [ ]:
V = critic(batch_obs).squeeze()
print("V.shape:", tuple(V.shape))
print("critic predictions (untrained, so near 0): mean=%.3f, std=%.3f" % (V.mean(), V.std()))

A_k = batch_rtgs - V.detach()     # .detach(): advantages are just numbers, no gradient flows through them
print("\nRaw advantage        : mean=%.3f, std=%.3f" % (A_k.mean(), A_k.std()))

A_k = (A_k - A_k.mean()) / (A_k.std() + 1e-10)
print("Normalised advantage : mean=%.3f, std=%.3f" % (A_k.mean(), A_k.std()))
print("first 5:", A_k[:5].numpy().round(3))

## 8. Step 4: UPDATE with the clipped objective

This is the heart of PPO.

**Ratio** = how much more (or less) likely the *new* policy finds the action compared to the *old* policy that collected it:

$$r_t = \frac{\pi_{new}(a_t|s_t)}{\pi_{old}(a_t|s_t)} = e^{\log\pi_{new} - \log\pi_{old}}$$

Two objectives:
- `surr1 = ratio * A`
- `surr2 = clip(ratio, 1-eps, 1+eps) * A`

We take the **minimum**. That removes any incentive to push the ratio outside `[0.8, 1.2]`, so the policy can't change too much in one iteration.
The loss is the *negative* mean because Adam minimises, but we want to maximise.

Before any update, new policy == old policy, so **ratio must be exactly 1**. Let's verify:

In [ ]:
mean = actor(batch_obs)
dist = MultivariateNormal(mean, cov_mat)
curr_log_probs = dist.log_prob(batch_acts)
ratios = torch.exp(curr_log_probs - batch_log_probs)

print("ratio before any update -> min=%.6f, max=%.6f (should both be 1.0)" % (ratios.min(), ratios.max()))

Now run the `n_updates_per_iteration` gradient steps on this one batch and watch how the ratio drifts away from 1:

In [ ]:
print(f"{'epoch':>5} | {'actor_loss':>10} | {'critic_loss':>11} | {'ratio min':>9} | {'ratio max':>9} | {'clipped %':>9}")
print("-" * 70)

for epoch in range(n_updates_per_iteration):
    V = critic(batch_obs).squeeze()
    mean = actor(batch_obs)
    dist = MultivariateNormal(mean, cov_mat)
    curr_log_probs = dist.log_prob(batch_acts)

    ratios = torch.exp(curr_log_probs - batch_log_probs)

    surr1 = ratios * A_k
    surr2 = torch.clamp(ratios, 1 - clip, 1 + clip) * A_k

    actor_loss = (-torch.min(surr1, surr2)).mean()
    critic_loss = nn.MSELoss()(V, batch_rtgs)

    actor_optim.zero_grad()
    actor_loss.backward(retain_graph=True)
    actor_optim.step()

    critic_optim.zero_grad()
    critic_loss.backward()
    critic_optim.step()

    # How many samples fell outside the clip range?
    clipped_frac = ((ratios - 1).abs() > clip).float().mean().item() * 100

    print(f"{epoch:>5} | {actor_loss.item():>10.4f} | {critic_loss.item():>11.2f} | "
          f"{ratios.min().item():>9.3f} | {ratios.max().item():>9.3f} | {clipped_frac:>8.1f}%")

## 9. Did that one iteration change anything?

Compare the critic's predictions with the real rewards-to-go, before vs after (the critic should now be closer):

In [ ]:
with torch.no_grad():
    V_after = critic(batch_obs).squeeze()
print("Critic MSE vs rewards-to-go after the update: %.2f" % nn.MSELoss()(V_after, batch_rtgs).item())
print("Critic predictions now: mean=%.2f (rewards-to-go mean=%.2f)" % (V_after.mean(), batch_rtgs.mean()))

## 10. The full training loop

Now we put sections 5-8 into a loop. First we **re-create** fresh networks so training starts from scratch
(the demo above already updated the old ones once).

Metrics are stored every iteration so we can plot them afterwards.
You can **interrupt the kernel (stop button)** at any time. The `try/except` keeps everything collected so far so the plots still work.

In [ ]:
# Fresh start
actor = nn.Sequential(
    nn.Linear(obs_dim, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, act_dim),
)
critic = nn.Sequential(
    nn.Linear(obs_dim, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
)
actor_optim = Adam(actor.parameters(), lr=lr)
critic_optim = Adam(critic.parameters(), lr=lr)

# History for plotting
hist_iter = []
hist_timesteps = []
hist_ep_return = []
hist_ep_len = []
hist_actor_loss = []
hist_critic_loss = []

t_so_far = 0
i_so_far = 0
delta_t = time.time_ns()

print(f"Learning... {max_timesteps_per_episode} timesteps per episode, "
      f"{timesteps_per_batch} per batch, {total_timesteps} total")

try:
    while t_so_far < total_timesteps:

        # ---------- 1. ROLLOUT ----------
        batch_obs, batch_acts, batch_log_probs, batch_rews, batch_lens = [], [], [], [], []
        t = 0
        while t < timesteps_per_batch:
            ep_rews = []
            obs, _ = env.reset()
            for ep_t in range(max_timesteps_per_episode):
                t += 1
                batch_obs.append(obs)

                mean = actor(torch.tensor(obs, dtype=torch.float))
                dist = MultivariateNormal(mean, cov_mat)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                action = action.detach().numpy()
                log_prob = log_prob.detach()

                obs, rew, terminated, truncated, _ = env.step(action)
                done = terminated | truncated

                ep_rews.append(rew)
                batch_acts.append(action)
                batch_log_probs.append(log_prob)
                if done:
                    break
            batch_lens.append(ep_t + 1)
            batch_rews.append(ep_rews)

        batch_obs = torch.tensor(np.array(batch_obs), dtype=torch.float)
        batch_acts = torch.tensor(np.array(batch_acts), dtype=torch.float)
        batch_log_probs = torch.tensor(np.array(batch_log_probs), dtype=torch.float)

        # ---------- 2. REWARDS-TO-GO ----------
        batch_rtgs = []
        for ep_rews in reversed(batch_rews):
            discounted_reward = 0
            for rew in reversed(ep_rews):
                discounted_reward = rew + discounted_reward * gamma
                batch_rtgs.insert(0, discounted_reward)
        batch_rtgs = torch.tensor(batch_rtgs, dtype=torch.float)

        t_so_far += np.sum(batch_lens)
        i_so_far += 1

        # ---------- 3. ADVANTAGE ----------
        V = critic(batch_obs).squeeze()
        A_k = batch_rtgs - V.detach()
        A_k = (A_k - A_k.mean()) / (A_k.std() + 1e-10)

        # ---------- 4. UPDATE ----------
        actor_losses, critic_losses, clipped_fracs = [], [], []
        for _ in range(n_updates_per_iteration):
            V = critic(batch_obs).squeeze()
            mean = actor(batch_obs)
            dist = MultivariateNormal(mean, cov_mat)
            curr_log_probs = dist.log_prob(batch_acts)

            ratios = torch.exp(curr_log_probs - batch_log_probs)
            surr1 = ratios * A_k
            surr2 = torch.clamp(ratios, 1 - clip, 1 + clip) * A_k

            actor_loss = (-torch.min(surr1, surr2)).mean()
            critic_loss = nn.MSELoss()(V, batch_rtgs)

            actor_optim.zero_grad()
            actor_loss.backward(retain_graph=True)
            actor_optim.step()

            critic_optim.zero_grad()
            critic_loss.backward()
            critic_optim.step()

            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())
            clipped_fracs.append(((ratios - 1).abs() > clip).float().mean().item() * 100)

        # ---------- 5. LOG ----------
        now = time.time_ns()
        iter_time = (now - delta_t) / 1e9
        delta_t = now

        avg_ep_len = np.mean(batch_lens)
        avg_ep_ret = np.mean([np.sum(r) for r in batch_rews])

        hist_iter.append(i_so_far)
        hist_timesteps.append(t_so_far)
        hist_ep_return.append(avg_ep_ret)
        hist_ep_len.append(avg_ep_len)
        hist_actor_loss.append(np.mean(actor_losses))
        hist_critic_loss.append(np.mean(critic_losses))

        print(f"Iter {i_so_far:4d} | steps {t_so_far:8d} | avg return {avg_ep_ret:8.2f} | avg len {avg_ep_len:6.1f} | "
              f"actor loss {np.mean(actor_losses): .4f} | critic loss {np.mean(critic_losses):9.2f} | "
              f"clipped {np.mean(clipped_fracs):4.1f}% | {iter_time:5.2f}s")

        # ---------- 6. SAVE ----------
        if i_so_far % save_freq == 0:
            torch.save(actor.state_dict(), './ppo_actor.pth')
            torch.save(critic.state_dict(), './ppo_critic.pth')
            print(f"          -> saved ppo_actor.pth and ppo_critic.pth (iteration {i_so_far})")

except KeyboardInterrupt:
    print("\nTraining interrupted by you. Keeping everything so far.")

print("\nDone. Iterations:", i_so_far, "| timesteps:", t_so_far)

## 11. Learning curves

- **Average episode return**: should rise toward roughly -200 to -150 (less negative = better)
- **Critic loss**: how well the critic predicts rewards-to-go
- **Actor loss**: hovers around 0 (it is normalised); it is *not* a progress measure like in supervised learning

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(hist_timesteps, hist_ep_return)
axes[0].set_title("Average episode return")
axes[0].set_xlabel("timesteps")
axes[0].set_ylabel("return")
axes[0].grid(alpha=0.3)

axes[1].plot(hist_timesteps, hist_critic_loss, color='tab:orange')
axes[1].set_title("Critic loss")
axes[1].set_xlabel("timesteps")
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

axes[2].plot(hist_timesteps, hist_actor_loss, color='tab:green')
axes[2].set_title("Actor loss")
axes[2].set_xlabel("timesteps")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Watch the trained actor

For evaluation we use the actor's **mean action directly** (no random noise), so the behaviour is deterministic.
Frames are drawn with matplotlib so this works inside Jupyter without a pop-up window.

In [ ]:
eval_returns = []
frames = []

for ep_num in range(3):
    obs, _ = env.reset()
    done = False
    ep_len, ep_ret = 0, 0
    while not done:
        if ep_num == 0 and ep_len % 10 == 0:     # keep a few frames of the first episode
            frames.append(env.render())
        action = actor(torch.tensor(obs, dtype=torch.float)).detach().numpy()
        obs, rew, terminated, truncated, _ = env.step(action)
        done = terminated | truncated
        ep_len += 1
        ep_ret += rew
    eval_returns.append(ep_ret)
    print(f"Episode #{ep_num} | length {ep_len} | return {ep_ret:.2f}")

print("\nAverage evaluation return: %.2f" % np.mean(eval_returns))

n_show = min(10, len(frames))
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2.2))
for ax, f, k in zip(np.atleast_1d(axes), frames[:n_show], range(n_show)):
    ax.imshow(f)
    ax.axis('off')
    ax.set_title(f"t={k * 10}", fontsize=9)
plt.show()

## 13. (Optional) Reload saved models later

Run this in a fresh kernel (after sections 0-3) to load what training saved, without retraining:

In [ ]:
# actor.load_state_dict(torch.load('./ppo_actor.pth'))
# critic.load_state_dict(torch.load('./ppo_critic.pth'))
# print("loaded")